# set 6

## 1) 데이터 및 시나리오

### 교육 수강자 분석

> 삼성전자 임직원의 체계적인 커리어 패스 설계를 위해 데이터 분석과 같은 Hard Skill 역량과 Soft Skill 역량을 조화롭게 함양하기 위해 HRD 부서의 조프로는 전산팀의 협조를 받아 데이터를 확보하였다.

※ 분석을 수행하기 전, 상기 데이터를 이용하여 아래의 전처리를 수행하시오.

- **단계 1:** 분석에 사용하지 않을 `city`, `company_size`, `company_type` 컬럼을 제거하시오.
- **단계 2:** 각 문자형 변수에 결측치가 하나라도 존재하는 행은 모두 제거하시오.
- **단계 3:** `experience` 변수 값이 `'>20'` 또는 `'<1'`이 있는 행을 제거하고 `experience` 변수의 유형을 정수로 변환하시오.
- **단계 4:** `last_new_job` 변수의 값이 `'>4'` 또는 `'never'`인 행을 제거하고 해당 변수의 유형을 정수로 변환하시오.

전처리 수행 이후 행 개수는 **7,522**이며 전처리가 완료된 데이터를 `base` 객체로 지정하고 이를 사용하여 문제를 풀이하시오.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `edu_enrollees.csv` | 19158 | 15 | UTF-8 |

---

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `enrollee_id` | int | 수료자 ID |
| `city` | string | 도시 코드 |
| `city_development_index` | float | 도시 발전 지표 |
| `gender` | string | 성별 |
| `relevant_experience` | string | 관련 분야 경험 여부 |
| `enrolled_university` | string | 수강 과목명 |
| `education_level` | string | 학력 |
| `major_discipline` | string | 전공 |
| `experience` | string | 경력 |
| `company_size` | string | 현 직장 직원 수 |

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `company_size` | string | 현 직장 직원 수 |
| `company_type` | string | 현 직장 유형 |
| `last_new_job` | string | 전 직장 근속연수 |
| `training_hours` | int | 수료 시간 |
| `target` | int | 전배 희망 여부 `(0: 비희망, 1: 희망)` |
| `Xgrp` | string | Train/Test Set 구분 |


## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.linear_model import LogisticRegression` |
| `from sklearn.neighbors import KNeighborsClassifier` |


In [64]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("../../dataset/edu_enrollees.csv")
#SCDI
display(df.shape)
display(df.columns)
display(df.dtypes)
display(df.isna().sum())

(19158, 15)

Index(['enrollee_id', 'city', 'city_development_index', 'gender',
       'relevant_experience', 'enrolled_university', 'education_level',
       'major_discipline', 'experience', 'company_size', 'company_type',
       'last_new_job', 'training_hours', 'target', 'Xgrp'],
      dtype='object')

enrollee_id               float64
city                       object
city_development_index    float64
gender                     object
relevant_experience        object
enrolled_university        object
education_level            object
major_discipline           object
experience                 object
company_size               object
company_type               object
last_new_job               object
training_hours            float64
target                    float64
Xgrp                       object
dtype: object

enrollee_id                  0
city                         0
city_development_index       0
gender                    4508
relevant_experience          0
enrolled_university        386
education_level            460
major_discipline          2813
experience                  65
company_size              5938
company_type              6140
last_new_job               423
training_hours               0
target                       0
Xgrp                         0
dtype: int64

In [65]:
#단계1
df_1 = df.drop(columns=['city', 'company_size', 'company_type']).copy()
display(df.shape, df_1.shape)
#단계2
cols_1 = df_1.select_dtypes('object').columns #dtype 별로 열 선택하는 방법
df_2 = df_1.dropna(subset=cols_1).copy()
display(df_2.shape)
#단계3
cond_1 = ~df_2['experience'].isin(['>20','<1'])
df_3 = df_2.loc[cond_1, :].copy()
df_3['experience'] = df_3['experience'].astype(int)
#단계4
cond_2 = ~df_3['last_new_job'].isin(['>4', 'never'])
df_4 = df_3.loc[cond_2, :].copy()
df_4['last_new_job'] = df_4['last_new_job'].astype(int)

base = df_4.copy()
display(base.shape)

(19158, 15)

(19158, 12)

(12477, 12)

(7522, 12)


---

### Q01.

`base`를 사용하여 관련 분야 경험 여부(`relevant_experience`)에 따른 전배 희망 여부(`target`)를 기술통계량으로 확인하고자 한다.

관련 분야 경험이 없는 수료자 중 전배를 희망하는 수료자의 비율을 **A**,  
관련 분야 경험이 있는 수료자 중 전배를 희망하는 수료자의 비율을 **B**라 할 때,  
**A/B를 구하시오.**

※ 관련 경험이 없는 사람은 `relevant_experience` 변수의 값이 `'No relevant experience'`인 사람으로 정의한다.  
※ 관련 경험이 있는 사람은 `relevant_experience` 변수의 값이 `'Has relevant experience'`인 사람으로 정의한다.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [66]:
df_q1 = base.copy()
display(df_q1['relevant_experience'].value_counts(), df_q1['target'].value_counts())
df_q1_gb = df_q1.groupby('relevant_experience')['target'].value_counts(normalize = True)
display(df_q1_gb)
A = df_q1_gb[('No relevant experience', 1.0)]
B = df_q1_gb[('Has relevant experience', 1.0)]
display(round(A/B, 2))

Has relevant experience    6109
No relevant experience     1413
Name: relevant_experience, dtype: int64

0.0    5662
1.0    1860
Name: target, dtype: int64

relevant_experience      target
Has relevant experience  0.0       0.784089
                         1.0       0.215911
No relevant experience   0.0       0.617127
                         1.0       0.382873
Name: target, dtype: float64

1.77


### Q02.

`base`를 사용하여 전배 희망 여부(`target`)에 영향을 주는 변수들을 확인하고자 한다.  
다음 절차에 따라 로지스틱 회귀분석을 수행하고 질문에 답하시오.

#### 단계 1

`gender`, `relevant_experience`, `enrolled_university`, `education_level`, `major_discipline` 변수로부터 더미 변수들을 생성한다.

단, 각 변수로부터 더미 변수를 생성할 때 마지막으로 등장하는 범주는 제외하도록 한다.

여기서 마지막으로 등장하는 범주란, 각 컬럼의 값을 사전 순으로 나열하였을 때 마지막으로 등장하는 값이다.

예를 들어, `col` 변수의 범주가 `['A', 'C', 'B', 'B', 'C', 'A']`의 값을 가진다면 사전 상의 마지막 값인 `C`가 제외된다.

#### 단계 2

단계 1에서 생성한 더미 변수와 `city_development_index`, `experience`, `last_new_job`, `training_hours`, `target`, `Xgrp` 변수를 결합하여 새로운 데이터셋을 구성한다.

- 데이터셋명: `job2`
- 이 데이터셋은 문제 3에서도 활용

이 때, `target`, `Xgrp`를 제외한 데이터셋의 컬럼은 아래 순서에 따르도록 한다.

1. `city_development_index`
2. `experience`
3. `last_new_job`
4. `training_hours`
5. `gender`의 더미 변수
6. `relevant_experience`의 더미 변수
7. `enrolled_university`의 더미 변수
8. `education_level`의 더미 변수
9. `major_discipline`의 더미 변수

#### 단계 3

단계 2에서 구성한 데이터셋 `job2`로 다음 조건에 따라 상수항(`Intercept`)이 포함된 로지스틱 회귀분석을 수행한다.

- 종속 변수: `target`
- 독립 변수(총 16개): `target`과 `Xgrp`를 제외한 나머지 변수
- 회귀식에 포함되는 독립 변수의 순서를 컬럼의 순서와 일치시킨다.

**상수항을 제외한 나머지 변수들에 대한 Odds Ratio 중 가장 큰 값을 기술하시오.**

$$
x_i \text{의 Odds Ratio}
=
\frac{
odds(P(Y=1|x_1,\cdots,x_i+1,\cdots,x_n))
}{
odds(P(Y=1|x_1,\cdots,x_i,\cdots,x_n))
}
$$

※ `LogisticRegression()` 클래스의 인자 `C`는 `100000`, `max_iter=1000`, `solver='liblinear'`으로 지정하시오.  
※ `LogisticRegression()` 클래스의 인자 `random_state`는 `123`으로 지정하시오.  
※ 정답은 소수점 셋째 자리에서 버림하여 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

---

In [67]:
df_q2 = base.copy()
#단계 1
#get_dummies로 우선 column을 만들고, columns를 삭제하는 식으로 해야한다.
#get_dummies 는 열 값에 대해 ohe을 수행하기때문에, 내가 임의로 값들을 지워버리면 dataframe의 행이 날아갈 수 있으니까
#각 열에 대해 조절해야함을 알자.
cols_dummies = [ 'gender','relevant_experience', 'enrolled_university', 'education_level', 'major_discipline']

ohe_list = [] #빈 dataframe에 concat이 아니라 한번에 dataframe list를 만들고 마지막에 붙이는거
for col in cols_dummies :
    ohe = pd.get_dummies(df_q2[col],prefix=col) #dataframe 한 열에 대해서 pd.get_dummies를 하면 prefix가 안 붙는다. dataframe에서 열을 선택할 때와 다르네
    display(ohe) #columns 이름이 공백이 있으면 _로 어떻게 바꿀 수 있나?
    ohe_drop=ohe.iloc[:,:-1].copy() #사전순인 것을 알기 때문에 좋은 방법이다.
    #응용 가능 - 사전순 맨 앞을 삭제하시오
    #응용 가능 - 사전역순으로 순서를 배치하싱오
    display(ohe_drop)
    ohe_list.append(ohe_drop)
display(ohe_list)
df_ohe = pd.concat(ohe_list,axis = 1)
display(df_ohe)

,gender_Female,gender_Male,gender_Other
8,0,1,0
11,0,1,0
19,1,0,0
20,0,1,0
21,0,1,0
...,...,...,...
19149,0,1,0
19150,1,0,0
19152,1,0,0
19153,0,1,0


,gender_Female,gender_Male
8,0,1
11,0,1
19,1,0
20,0,1
21,0,1
...,...,...
19149,0,1
19150,1,0
19152,1,0
19153,0,1


,relevant_experience_Has relevant experience,relevant_experience_No relevant experience
8,1,0
11,1,0
19,1,0
20,1,0
21,1,0
...,...,...
19149,1,0
19150,1,0
19152,1,0
19153,0,1


,relevant_experience_Has relevant experience
8,1
11,1
19,1
20,1
21,1
...,...
19149,1
19150,1
19152,1
19153,0


,enrolled_university_Full time course,enrolled_university_Part time course,enrolled_university_no_enrollment
8,0,0,1
11,0,0,1
19,0,0,1
20,0,0,1
21,0,0,1
...,...,...,...
19149,0,0,1
19150,0,0,1
19152,0,0,1
19153,0,0,1


...

,education_level_Graduate,education_level_Masters
8,1,0
11,1,0
19,1,0
20,0,1
21,0,1
...,...,...
19149,0,1
19150,1,0
19152,1,0
19153,1,0


,major_discipline_Arts,major_discipline_Business Degree,major_discipline_Humanities,major_discipline_No Major,major_discipline_Other,major_discipline_STEM
8,0,0,0,0,0,1
11,0,0,0,0,0,1
19,1,0,0,0,0,0
20,0,0,0,0,0,1
21,0,0,0,0,0,1
...,...,...,...,...,...,...
19149,0,0,0,0,0,1
19150,0,0,0,0,0,1
19152,0,0,1,0,0,0
19153,0,0,1,0,0,0


,major_discipline_Arts,major_discipline_Business Degree,major_discipline_Humanities,major_discipline_No Major,major_discipline_Other
8,0,0,0,0,0
11,0,0,0,0,0
19,1,0,0,0,0
20,0,0,0,0,0
21,0,0,0,0,0
...,...,...,...,...,...
19149,0,0,0,0,0
19150,0,0,0,0,0
19152,0,0,1,0,0
19153,0,0,1,0,0


[       gender_Female  gender_Male
 8                  0            1
 11                 0            1
 19                 1            0
 20                 0            1
 21                 0            1
 ...              ...          ...
 19149              0            1
 19150              1            0
 19152              1            0
 19153              0            1
 19154              0            1
 
 [7522 rows x 2 columns],
        relevant_experience_Has relevant experience
 8                                                1
 11                                               1
 19                                               1
 20                                               1
 21                                               1
 ...                                            ...
 19149                                            1
 19150                                            1
 19152                                            1
 19153                          

,gender_Female,gender_Male,relevant_experience_Has relevant experience,enrolled_university_Full time course,enrolled_university_Part time course,education_level_Graduate,education_level_Masters,major_discipline_Arts,major_discipline_Business Degree,major_discipline_Humanities,major_discipline_No Major,major_discipline_Other
8,0,1,1,0,0,1,0,0,0,0,0,0
11,0,1,1,0,0,1,0,0,0,0,0,0
19,1,0,1,0,0,1,0,1,0,0,0,0
20,0,1,1,0,0,0,1,0,0,0,0,0
21,0,1,1,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
19149,0,1,1,0,0,0,1,0,0,0,0,0
19150,1,0,1,0,0,1,0,0,0,0,0,0
19152,1,0,1,0,0,1,0,0,0,1,0,0
19153,0,1,0,0,0,1,0,0,0,1,0,0


In [68]:
#단계 2
cols_q2_2 = ['city_development_index', 'experience',  'last_new_job',  'training_hours']
cols_q2_3 = ['target', 'Xgrp']
df_list_q2 = [df_q2[cols_q2_2],df_ohe,df_q2[cols_q2_3]]
job2 = pd.concat(df_list_q2,axis=1)
display(job2.columns)

Index(['city_development_index', 'experience', 'last_new_job',
       'training_hours', 'gender_Female', 'gender_Male',
       'relevant_experience_Has relevant experience',
       'enrolled_university_Full time course',
       'enrolled_university_Part time course', 'education_level_Graduate',
       'education_level_Masters', 'major_discipline_Arts',
       'major_discipline_Business Degree', 'major_discipline_Humanities',
       'major_discipline_No Major', 'major_discipline_Other', 'target',
       'Xgrp'],
      dtype='object')

In [69]:
#단계3
display(job2.shape)
#D
X = job2.drop(columns=['target','Xgrp'])
y = job2['target']
#N
#M
model = LogisticRegression(C=100000,
                           max_iter=1000,
                           solver='liblinear',
                           random_state = 123)
model.fit(X,y)
display(model.coef_[0])

ser = pd.Series(np.exp(model.coef_[0]), index=X.columns)
display(ser, ser.max(), ser.idxmax())
display(np.floor(ser.max()*100)/100)
#E

(7522, 18)

array([-6.16042356e+00, -2.84523178e-02,  9.51256118e-02, -9.32104029e-04,
       -1.77221690e-01, -1.43611016e-01, -7.67253194e-01,  5.15739072e-01,
       -2.82857983e-01,  2.58989370e-01, -4.40244893e-02,  2.90013957e-01,
        1.18976977e-01,  2.47072131e-01,  4.04528542e-01, -4.45273205e-01])

city_development_index                         0.002111
experience                                     0.971949
last_new_job                                   1.099797
training_hours                                 0.999068
gender_Female                                  0.837594
gender_Male                                    0.866225
relevant_experience_Has relevant experience    0.464287
enrolled_university_Full time course           1.674876
enrolled_university_Part time course           0.753627
education_level_Graduate                       1.295620
education_level_Masters                        0.956931
major_discipline_Arts                          1.336446
major_discipline_Business Degree               1.126344
major_discipline_Humanities                    1.280271
major_discipline_No Major                      1.498596
major_discipline_Other                         0.640649
dtype: float64

1.6748758983036531

'enrolled_university_Full time course'

1.67

### Q03.

`job2`를 이용하여 전체 데이터를 Train과 Test Set으로 나누고, Train Set으로 학습한 모델을 Test Set에 적용하여 모델을 평가하고자 한다.

다음 절차에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

2단계에서 구성한 데이터셋 `job2`에서 `Xgrp` 컬럼의 값이 `'train'`인 경우 Train Set으로, `'test'`인 경우 Test Set으로 정의하여 분할한다.

#### 단계 2

아래 가이드에 따라 Train Set으로 K-NN 분류 모델을 학습하고, 이 모델을 Test Set에 적용한다.

- 종속 변수: 전배 희망 여부(`target`)
- 독립 변수(총 16개): 전배 희망 여부(`target`)와 Train/Test set 구분 변수(`Xgrp`)를 제외한 모든 변수
- Euclidean 거리 기준 가장 가까운 5개 데이터의 전배 희망 여부(`target`)를 활용하여 예측

#### 단계 3

예측 결과를 바탕으로 아래 정의된 지표 **A**를 계산하여 기술하시오.

$$
A =
\frac{
(\# \text{ of true positive}) + (\# \text{ of true negative})
}{
(\# \text{ of total data})
}
$$

※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [78]:
df_q3 = job2.copy()

train = df_q3.loc[df_q3['Xgrp'] == 'train', :].copy()
test = df_q3.loc[df_q3['Xgrp'] == 'test', :].copy()

#D
train_X = train.drop(columns=['target','Xgrp'])
train_y = train['target']

test_X = test.drop(columns=['target','Xgrp'])
test_y = test['target']

#N

#M
model = KNeighborsClassifier(n_neighbors = 5)
model.fit(train_X, train_y)
pred_y = model.predict(test_X)
#E
df_q3_ct = pd.crosstab(test_y, pred_y)
display(df_q3_ct)
A = (df_q3_ct[0][0] + df_q3_ct[1][1])/df_q3_ct.sum().sum()
display(round(A,2))

col_0,0.0,1.0
target,,
0.0,1897,195
1.0,616,108


0.71